# All-sky cloud condition + diffuse radiation — Colab GPU training

Trains `allsky.SkyFusionNet` on all-sky camera frames paired with LabMiM radiation sensors.

**Runtime → Change runtime type → GPU (T4 is fine)** before running.

In [ ]:
!nvidia-smi -L
%pip install -q "labmim-micrometeorology[allsky] @ git+https://github.com/Bruno-Mascarenhas/micrometeorology.git"

## Data
Point the paths at your Drive (or upload directly). You need:
- all-sky videos `allsky-YYYYMMDD.mp4`
- the Campbell `LBM_lenta` `.dat` file(s) covering the same dates

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

VIDEOS = "/content/drive/MyDrive/labmim/all-sky"  # allsky-*.mp4
SENSORS = "/content/drive/MyDrive/labmim/LBM_lenta_2025.dat"
WORKDIR = "/content/allsky"

In [ ]:
import pathlib

config_yaml = f"""
sensor:
  paths: ["{SENSORS}"]
  diffuse_column: "PSP_Wm2_Avg"   # live diffuse sensor (CMP21 W/m2 channel is currently zero-filled)
video:
  start_time: "06:00"             # local time of frame 0 — match the camera schedule
train:
  epochs: 30
  batch_size: 64
  num_workers: 2
  device: "auto"                  # resolves to cuda on Colab GPU
  amp: true                       # mixed precision on T4/L4/A100
  out_dir: "{WORKDIR}/run"
"""

pathlib.Path(WORKDIR).mkdir(parents=True, exist_ok=True)
pathlib.Path(f"{WORKDIR}/config.yaml").write_text(config_yaml)
print(config_yaml)

In [ ]:
# 1) Extract frames (one JPEG per timelapse frame = one minute of sky)
import glob
import subprocess

for video in sorted(glob.glob(f"{VIDEOS}/allsky-*.mp4")):
    subprocess.run(
        [
            "allsky",
            "extract-frames",
            video,
            "-o",
            f"{WORKDIR}/frames",
            "--config",
            f"{WORKDIR}/config.yaml",
        ],
        check=True,
    )

In [ ]:
# 2) Pair frames with sensor rows (5-minute tolerance, daytime only)
!allsky build-index --manifest {WORKDIR}/frames/manifest.parquet \
    --out {WORKDIR}/run/index.parquet --config {WORKDIR}/config.yaml

In [ ]:
# 3) Live metrics while training
%load_ext tensorboard
%tensorboard --logdir {WORKDIR}/run/runs

In [ ]:
# 4) Train (resumable: add --resume {WORKDIR}/run/last.pt after an interruption)
!allsky train --config {WORKDIR}/config.yaml --index {WORKDIR}/run/index.parquet

Artifacts land in `run/`: `best.pt` / `last.pt` checkpoints (model + optimizer + config),
`config.json`, `metadata.json` (git commit, versions, timings), TensorBoard events under `runs/`.
Copy `run/` back to Drive to persist across Colab sessions.